In [58]:
# ============================================================
# EXERCISE 1: PROMPT CHAINING FOR CUSTOMER SUPPORT AI
# Tools: Google Colab, Python, Gemini API
# Goal: Use a 3-step prompt chain where each AI response
# becomes context for the next step.
# ============================================================


# --- SETUP ---

# Install the Gemini library
!pip install -q -U google-generativeai

# Import required libraries
import google.generativeai as genai
from google.colab import userdata
import time


# Retrieve Gemini API key securely from Google Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")

    if not GOOGLE_API_KEY:
        raise ValueError("GEMINI_API_KEY was not found.")

    genai.configure(api_key=GOOGLE_API_KEY)
    print("Gemini API successfully configured.")

except Exception as e:
    print(f"Error configuring Gemini API: {e}")
    print("Make sure your Colab Secret is named GEMINI_API_KEY.")
    GOOGLE_API_KEY = None


# Initialize Gemini model
model = None

if GOOGLE_API_KEY:
    try:
        model = genai.GenerativeModel("gemini-2.5-flash")
        print("Using Gemini model: gemini-2.5-flash")
    except Exception as e:
        print(f"Error initializing Gemini model: {e}")


# Function for calling Gemini with basic retry handling
def call_gemini_api(prompt, max_retries=5, initial_delay=1):

    if not model:
        return "Error: Gemini model is not configured."

    for attempt in range(max_retries):

        try:
            response = model.generate_content(prompt)

            # Short pause between API calls
            time.sleep(initial_delay)

            return response.text

        except Exception as e:

            # Retry temporary errors without printing them
            if attempt < max_retries - 1:
                time.sleep(initial_delay * (2 ** attempt))

            else:
                print(f"API call failed after {max_retries} attempts: {e}")
                return f"Error after multiple retries: {e}"


# ============================================================
# SAMPLE CUSTOMER EMAIL
# ============================================================

customer_email = """
Subject: Refund still missing after return

Hi Customer Support,

I returned a pair of headphones about two weeks ago and received an
email confirming that the return was accepted, but I still have not
received the refund on my card. I checked my account and do not see
any update on the payment status.

I also tried contacting support yesterday but was transferred between
departments without getting an answer. I would really appreciate some
help figuring out what happened and when I should expect the refund.

Thanks,
Alex
"""

print("\nRAW CUSTOMER EMAIL:")
print(customer_email)
print("\n" + "=" * 60 + "\n")


# ============================================================
# STEP 1: CLASSIFICATION AND PRIORITY
# ============================================================

print("--- STEP 1 OUTPUT: CLASSIFICATION ---")

step1_prompt = f"""
Analyze the following customer email.

Complete these tasks:

1. Classify the issue into ONE of these categories:
   - Billing
   - Technical Support
   - Returns
   - General Inquiry

2. Assign a priority level:
   - High
   - Medium
   - Low

3. Write a one-sentence summary of the customer's main concern.

Customer Email:
{customer_email}

Return the response exactly in this format:

Category: [category]
Priority: [priority]
Summary: [one-sentence summary]
"""

step1_output = call_gemini_api(step1_prompt)

print(step1_output)
print("\n" + "=" * 60 + "\n")


# ============================================================
# STEP 2: INFORMATION EXTRACTION AND ESCALATION
# ============================================================

print("--- STEP 2 OUTPUT: EXTRACTION & TRIAGE ---")

step2_prompt = f"""
Using BOTH the original customer email and the Step 1 classification,
analyze what information is still needed and whether the issue
requires human escalation.

Complete these tasks:

1. Identify any missing information that may be needed to resolve
   the customer's problem.

2. Determine whether human escalation is required.

Consider escalation when:
- The customer has already made unsuccessful attempts to get support.
- Important account or order information is missing.
- The issue is high priority.
- The issue requires access to company or customer account systems.
- A human employee would need to investigate the issue.

3. Give a one-sentence explanation for the escalation decision.

Original Customer Email:
{customer_email}

Step 1 Classification:
{step1_output}

Return the response exactly in this format:

Missing Info: [missing information or None]
Human Escalation Required: YES/NO
Escalation Reason: [one-sentence explanation]
"""

step2_output = call_gemini_api(step2_prompt)

print(step2_output)
print("\n" + "=" * 60 + "\n")


# ============================================================
# STEP 3: CUSTOMER RESPONSE GENERATION
# ============================================================

print("--- STEP 3 OUTPUT: FINAL RESPONSE ---")

step3_prompt = f"""
Write a customer-facing response using ALL of the information below:

- The original customer email
- The Step 1 classification
- The Step 2 information extraction and escalation analysis

Follow these constraints:

- Use a professional, calm, and empathetic tone.
- Acknowledge the customer's frustration about the delayed refund.
- Acknowledge that the customer was previously transferred between
  departments without getting an answer.
- If Step 2 recommends escalation, explain that the issue requires
  additional support from a human representative.
- Request any missing information identified in Step 2.
- Do NOT ask for passwords, full credit card numbers, Social Security
  numbers, or other sensitive information.
- Do NOT use overly generic phrases such as "Dear Valued Customer."
- Do NOT claim that an employee has already reviewed, escalated,
  flagged, or assigned the case.
- Do NOT invent company policies.
- Do NOT guarantee that the customer will receive a refund.
- Do NOT guarantee a specific refund date.
- Keep the response under 150 words.

Original Customer Email:
{customer_email}

Step 1 Classification:
{step1_output}

Step 2 Triage Output:
{step2_output}

Customer Response:
"""

step3_output = call_gemini_api(step3_prompt)

print(step3_output)
print("\n" + "=" * 60 + "\n")


# ============================================================
# ITERATION EVIDENCE
# ============================================================

print("--- ITERATION EVIDENCE ---")

original_prompt = """
Write a response to the customer based on their email,
classification, and triage.
"""

print("\nOriginal Step 3 Prompt:")
print(original_prompt)

print("""
Testing Observation:
The original prompt was too broad and gave the AI too much freedom.
The response could become generic, robotic, or fail to clearly
acknowledge the customer's frustration and provide appropriate
next steps.

Prompt Improvement:
The revised Step 3 prompt added specific requirements for:
- Professional and empathetic tone
- Acknowledging the delayed refund
- Acknowledging the previous unsuccessful support experience
- Requesting missing information
- Handling human escalation
- Avoiding sensitive information
- Avoiding unsupported company policies or guarantees
- Keeping the response under 150 words

Result:
The improved prompt gives the AI clearer instructions and produces
a more focused, helpful, and appropriate customer support response.
""")


# ============================================================
# PROMPT CHAIN SUMMARY
# ============================================================

print("=" * 60)
print("PROMPT CHAIN COMPLETED")
print("=" * 60)

print("""
Customer Email
      ↓
Step 1: Classification & Priority
      ↓
Step 2: Information Extraction & Escalation
      ↓
Step 3: Final Customer Response
""")

Gemini API successfully configured.
Using Gemini model: gemini-2.5-flash

RAW CUSTOMER EMAIL:

Subject: Refund still missing after return

Hi Customer Support,

I returned a pair of headphones about two weeks ago and received an
email confirming that the return was accepted, but I still have not
received the refund on my card. I checked my account and do not see
any update on the payment status.

I also tried contacting support yesterday but was transferred between
departments without getting an answer. I would really appreciate some
help figuring out what happened and when I should expect the refund.

Thanks,
Alex



--- STEP 1 OUTPUT: CLASSIFICATION ---
Category: Billing
Priority: High
Summary: The customer has not received a refund for a confirmed return two weeks ago and is frustrated after an unsuccessful attempt to contact support regarding the missing payment.


--- STEP 2 OUTPUT: EXTRACTION & TRIAGE ---
Missing Info: Order number, Return Authorization (RMA) or Return ID, and th